In [117]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
from pyspark.sql.functions import col, count, when

In [118]:


spark = SparkSession.builder.appName("Spark SQL").getOrCreate()
df=spark.read.parquet("clickstream_raw.parquet")
df.printSchema()
df.show(5)

root
 |-- event_id: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- page: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- event_time: timestamp_ntz (nullable = true)
 |-- time_on_page_s: long (nullable = true)
 |-- device: string (nullable = true)
 |-- referrer: string (nullable = true)
 |-- country: string (nullable = true)

+--------+----------+---------+----------+--------+----------+-------------------+--------------+-------+--------+-------+
|event_id|session_id|  user_id|event_type|    page|product_id|         event_time|time_on_page_s| device|referrer|country|
+--------+----------+---------+----------+--------+----------+-------------------+--------------+-------+--------+-------+
|       0| sess_7271| user_495| page_view|    home|      NULL|2024-03-11 05:36:35|             0|Desktop|  direct|UNKNOWN|
|       1| sess_7604|user_2626| page_view|    cart|  

In [119]:
print(f"Total rows: {df.count()}")
print(f"unique session_id: {df.select('event_id').distinct().count()}")

Total rows: 50500
unique session_id: 50000


In [120]:
df=df.dropDuplicates(["event_id"])
df=df.filter(f.col("time_on_page_s") > 0)


In [121]:
df = df.withColumn("device",   f.lower(f.trim(f.col("device")))) \
       .withColumn("referrer", f.lower(f.trim(f.col("referrer")))) \
       .withColumn("country",  f.lower(f.trim(f.col("country"))))

df=df.withColumn("is_outlier",f.when(f.col("time_on_page_s")>3600,True).otherwise(False))
df.show(5)

+--------+----------+---------+----------+--------+----------+-------------------+--------------+-------+--------+-------+----------+
|event_id|session_id|  user_id|event_type|    page|product_id|         event_time|time_on_page_s| device|referrer|country|is_outlier|
+--------+----------+---------+----------+--------+----------+-------------------+--------------+-------+--------+-------+----------+
|      19|  sess_131|user_2100| page_view| product|      NULL|2024-01-27 13:28:27|           300| mobile|  google|     au|     false|
|      25| sess_7582|     NULL|  purchase| product|  prod_351|2024-01-17 03:25:14|           300| mobile|   email|   NULL|     false|
|      26| sess_6950|     NULL|  purchase| product|  prod_212|2024-03-23 15:46:54|            10|desktop|  google|     au|     false|
|      29| sess_5052|user_1992|    search|checkout|  prod_332|2024-01-21 10:03:59|            10| mobile|  google|   NULL|     false|
|      31| sess_1185|user_2748|    search|checkout|      NULL|

In [122]:
session_window = Window.partitionBy("session_id").orderBy("event_time")
df=df.withColumn("session_seq_num",f.row_number().over(session_window))
df.show(5)

+--------+----------+---------+----------------+------------+----------+-------------------+--------------+------+--------+-------+----------+---------------+
|event_id|session_id|  user_id|      event_type|        page|product_id|         event_time|time_on_page_s|device|referrer|country|is_outlier|session_seq_num|
+--------+----------+---------+----------------+------------+----------+-------------------+--------------+------+--------+-------+----------+---------------+
|   24093|    sess_1|user_1033|     add_to_cart|      search|  prod_262|2024-01-20 03:55:55|           120|mobile|  social|     au|     false|              1|
|   38407|    sess_1| user_369|        purchase|     product|   prod_89|2024-03-07 05:11:42|          9999|mobile|  google|     au|      true|              2|
|   41881|    sess_1| user_352|     add_to_cart|confirmation|  prod_479|2024-03-19 18:33:30|            10|  NULL|  google|     au|     false|              3|
|   26554| sess_1000|     NULL|remove_from_car

In [123]:
# Step 1 — get all event types per session as a set
session_events = df.filter(f.col("event_type") != "NULL") \
                   .groupBy("session_id") \
                   .agg(f.collect_set("event_type").alias("event_types"))

session_events.show(5, truncate=False)

# Step 2 — flag abandoned sessions
session_events = session_events.withColumn("is_abandoned",
    f.when(
        f.array_contains(f.col("event_types"), "add_to_cart") &
        ~f.array_contains(f.col("event_types"), "purchase"),
        True
    ).otherwise(False)
)

df = df.join(
    session_events.select("session_id", "is_abandoned"),
    on="session_id",
    how="left"
)
df.show(5)

+----------+------------------------------------+
|session_id|event_types                         |
+----------+------------------------------------+
|sess_1    |[purchase, add_to_cart]             |
|sess_10   |[purchase, remove_from_cart, search]|
|sess_100  |[search]                            |
|sess_1000 |[abandon, remove_from_cart]         |
|sess_1001 |[page_view]                         |
+----------+------------------------------------+
only showing top 5 rows
+----------+--------+---------+----------------+------------+----------+-------------------+--------------+------+--------+-------+----------+---------------+------------+
|session_id|event_id|  user_id|      event_type|        page|product_id|         event_time|time_on_page_s|device|referrer|country|is_outlier|session_seq_num|is_abandoned|
+----------+--------+---------+----------------+------------+----------+-------------------+--------------+------+--------+-------+----------+---------------+------------+
|    sess_

In [ ]:
df = df.withColumn("session_id", f.regexp_extract(f.col("session_id"), r"_(.+)$", 1)) \
       .withColumn("user_id",    f.regexp_extract(f.col("user_id"),    r"_(.+)$", 1)) \
       .withColumn("product_id",    f.regexp_extract(f.col("product_id"),    r"_(.+)$", 1)) 


In [125]:
df = df.withColumn("event_date", f.to_date(f.col("event_time"))) \
       .withColumn("event_time", f.date_format(f.col("event_time"), "HH:mm:ss"))

In [126]:
df.show(5)

+----------+--------+-------+----------------+------------+----------+----------+--------------+------+--------+-------+----------+---------------+------------+----------+
|session_id|event_id|user_id|      event_type|        page|product_id|event_time|time_on_page_s|device|referrer|country|is_outlier|session_seq_num|is_abandoned|event_date|
+----------+--------+-------+----------------+------------+----------+----------+--------------+------+--------+-------+----------+---------------+------------+----------+
|         1|   24093|   1033|     add_to_cart|      search|       262|  03:55:55|           120|mobile|  social|     au|     false|              1|       false|2024-01-20|
|         1|   38407|    369|        purchase|     product|        89|  05:11:42|          9999|mobile|  google|     au|      true|              2|       false|2024-03-07|
|         1|   41881|    352|     add_to_cart|confirmation|       479|  18:33:30|            10|  NULL|  google|     au|     false|         